# Auditoria de reprodutibilidade e encerramento

Este notebook verifica a identidade dos protocolos, manifestos e resultados finais. Os hashes dos YAMLs são obtidos da revisão histórica indicada no README. Se o projeto foi extraído de um ZIP, informe em `HISTORICAL_PROJECT_ROOT` a raiz de uma cópia extraída da revisão histórica.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/Projeto_TCC_Lucas')
OUTPUT = PROJECT_ROOT / 'output/reproducibility_audit'
HISTORICAL_PROJECT_ROOT = None
%cd {PROJECT_ROOT}

## 1. Testes do auditor
Os testes abaixo verificam as funções de hash e contagem sem acessar imagens, máscaras ou checkpoints.

In [ ]:
import os, subprocess, sys

env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT / "src")
subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_reproducibility_audit.py"],
    check=True,
    env=env,
)

## 2. Auditoria final
A auditoria confere presença, hashes, run IDs e cardinalidades das tabelas finais. Checkpoints grandes permanecem como dependências externas identificadas por SHA-256.

In [ ]:
command = [
    sys.executable,
    "scripts/run_reproducibility_audit.py",
    "--project-root",
    str(PROJECT_ROOT),
    "--output-dir",
    str(OUTPUT),
]
if HISTORICAL_PROJECT_ROOT is not None:
    command.extend(["--historical-project-root", str(HISTORICAL_PROJECT_ROOT)])
subprocess.run(command, check=True, env=env)

## 3. Resultado legível
`PASS` indica evidência local verificada. `EXTERNAL_REQUIRED` identifica arquivos grandes mantidos no Drive. `DECLARED_LIMITATION` registra limites metodológicos sem escondê-los.

In [ ]:
import json, pandas as pd

display(pd.read_csv(OUTPUT / "reproducibility_checklist.csv"))
print(
    json.dumps(
        json.loads((OUTPUT / "day18_audit_report.json").read_text(encoding="utf-8")),
        indent=2,
        ensure_ascii=False,
    )
)